# Transformer Forecasting

## Energy Time-Series Transfer Learning Project

This notebook introduces a compact **Transformer Encoder** in PyTorch using
the same one-hour-ahead, 24-hour lookback setup as Notebook 02.

- Source task: PV generation forecasting
- Target task: household grid-import forecasting
- Framework: PyTorch
- Forecast horizon: next hour
- Input window: previous 24 hours
- Evaluation: chronological train/validation/test split

The data were prepared in Notebook 01 from cumulative energy measurements and
are expressed as hourly energy in kWh.


## Why a Transformer?

Transformers use attention to learn relationships between positions in a
sequence. Here, a compact Transformer Encoder is used as the next model after
the LSTM baseline.

The objective is not to build a large model, but to establish a clean
Transformer forecasting baseline that can later be adapted for transfer
learning.

**Notebook 02:** LSTM  
↓  
**Notebook 03:** Transformer  
↓  
**Notebook 04:** Transfer learning: PV → grid import


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

print("PyTorch version:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
DATA_PATH = "/content/residential4_model_data.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

data = df[[
    "timestamp",
    "pv_hourly",
    "grid_import_hourly"
]].copy()

print("Shape:", data.shape)
print("Start:", data["timestamp"].min())
print("End:", data["timestamp"].max())
print("\nMissing values:")
print(data.isna().sum())


## 1. Chronological train/validation/test split

The same 70/15/15 chronological split from Notebook 02 is used. The series is
never randomly shuffled before splitting.


In [ ]:
n = len(data)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = data.iloc[:train_end].copy()
val = data.iloc[train_end:val_end].copy()
test = data.iloc[val_end:].copy()

print("Train:", train["timestamp"].min(), "→", train["timestamp"].max())
print("Validation:", val["timestamp"].min(), "→", val["timestamp"].max())
print("Test:", test["timestamp"].min(), "→", test["timestamp"].max())

print("\nSizes:")
print("Train:", len(train))
print("Validation:", len(val))
print("Test:", len(test))


In [ ]:
def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mean_target = np.mean(y_true)
    nrmse = rmse / mean_target if mean_target != 0 else np.nan
    return {"MAE": mae, "RMSE": rmse, "nRMSE": nrmse}


def print_metrics(name, y_true, y_pred):
    m = regression_metrics(y_true, y_pred)
    print(name)
    print("-" * len(name))
    print(f"MAE:   {m['MAE']:.4f} kWh")
    print(f"RMSE:  {m['RMSE']:.4f} kWh")
    print(f"nRMSE: {m['nRMSE']:.2%}")


## 2. Persistence baseline

For a one-hour-ahead forecast, persistence predicts that the next hour will
equal the current hour.

The first test prediction uses the final observation from validation.


In [ ]:
def persistence_forecast(train_val_series, test_series):
    train_val_series = np.asarray(train_val_series)
    test_series = np.asarray(test_series)

    previous_values = np.concatenate([
        [train_val_series[-1]],
        test_series[:-1]
    ])

    return test_series, previous_values


pv_persistence_true, pv_persistence_pred = persistence_forecast(
    val["pv_hourly"].values,
    test["pv_hourly"].values
)

grid_persistence_true, grid_persistence_pred = persistence_forecast(
    val["grid_import_hourly"].values,
    test["grid_import_hourly"].values
)

print_metrics("Persistence – PV test set",
              pv_persistence_true, pv_persistence_pred)

print()

print_metrics("Persistence – Grid-import test set",
              grid_persistence_true, grid_persistence_pred)


## 3. Sequence preparation

The model receives the previous **24 hourly observations** and predicts the
next hour.

Scaling is fitted using the training period only. Validation and test
sequences include the preceding 24 hours as historical context.


In [ ]:
LOOKBACK = 24

def make_sequences(values, lookback=24):
    values = np.asarray(values)
    X, y = [], []

    for i in range(lookback, len(values)):
        X.append(values[i-lookback:i])
        y.append(values[i])

    return np.array(X), np.array(y)


## 4. Transformer architecture

The model uses:

1. Linear input projection
2. Sinusoidal positional encoding
3. Transformer Encoder
4. Mean pooling across the 24-hour sequence
5. Feed-forward prediction head

This is intentionally compact so that the experiment remains fast in
Google Colab.


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()

        position = torch.arange(max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-np.log(10000.0) / d_model)
        )

        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(
            position * div_term[:pe[:, 1::2].shape[1]]
        )

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TimeSeriesTransformer(nn.Module):
    def __init__(
        self,
        input_size=1,
        d_model=64,
        nhead=4,
        num_layers=2,
        dim_feedforward=128,
        dropout=0.1
    ):
        super().__init__()

        self.input_projection = nn.Linear(input_size, d_model)

        self.positional_encoding = PositionalEncoding(
            d_model=d_model,
            max_len=LOOKBACK
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.head = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        x = self.input_projection(x)
        x = self.positional_encoding(x)
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.head(x).squeeze(-1)


example_model = TimeSeriesTransformer().to(DEVICE)

print(example_model)

print(
    f"\nTrainable parameters: "
    f"{sum(p.numel() for p in example_model.parameters()):,}"
)


In [ ]:
def run_transformer_experiment(
    dataframe,
    target,
    lookback=24,
    epochs=30,
    batch_size=64,
    learning_rate=1e-3
):
    values = dataframe[target].values.astype(np.float32)

    n = len(values)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    scaler = StandardScaler()
    scaler.fit(values[:train_end].reshape(-1, 1))

    scaled = scaler.transform(
        values.reshape(-1, 1)
    ).flatten().astype(np.float32)

    X_train, y_train = make_sequences(
        scaled[:train_end], lookback
    )

    X_val, y_val = make_sequences(
        scaled[train_end-lookback:val_end], lookback
    )

    X_test, y_test = make_sequences(
        scaled[val_end-lookback:], lookback
    )

    X_train = X_train[..., np.newaxis]
    X_val = X_val[..., np.newaxis]
    X_test = X_test[..., np.newaxis]

    train_dataset = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32)
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    X_val_tensor = torch.tensor(X_val, dtype=torch.float32, device=DEVICE)
    y_val_tensor = torch.tensor(y_val, dtype=torch.float32, device=DEVICE)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32, device=DEVICE)

    model = TimeSeriesTransformer().to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-4
    )

    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    best_state = None
    patience = 5
    patience_counter = 0

    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()

            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()
            train_losses.append(loss.item())

        train_loss = np.mean(train_losses)

        model.eval()
        with torch.no_grad():
            val_predictions = model(X_val_tensor)
            val_loss = criterion(
                val_predictions,
                y_val_tensor
            ).item()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"Train loss: {train_loss:.4f} | "
            f"Val loss: {val_loss:.4f}"
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping.")
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        y_pred_scaled = model(X_test_tensor).cpu().numpy()

    y_true = scaler.inverse_transform(
        y_test.reshape(-1, 1)
    ).flatten()

    y_pred = scaler.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    ).flatten()

    return model, history, y_true, y_pred


# 5. PV Transformer forecasting

PV generation is the **source task** for the later transfer-learning
experiment.


In [ ]:
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

pv_transformer, pv_transformer_history, pv_transformer_true, pv_transformer_pred = (
    run_transformer_experiment(
        data,
        target="pv_hourly",
        lookback=LOOKBACK,
        epochs=30
    )
)

print()

print_metrics(
    "Transformer – PV test set",
    pv_transformer_true,
    pv_transformer_pred
)


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    pv_transformer_history["train_loss"],
    label="Training loss"
)

plt.plot(
    pv_transformer_history["val_loss"],
    label="Validation loss"
)

plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("Transformer Training History – PV")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
pv_test_timestamps = test["timestamp"].reset_index(drop=True)
plot_n = min(168, len(pv_transformer_true))

plt.figure(figsize=(15, 5))

plt.plot(
    pv_test_timestamps.iloc[:plot_n],
    pv_transformer_true[:plot_n],
    label="Actual",
    linewidth=1.2
)

plt.plot(
    pv_test_timestamps.iloc[:plot_n],
    pv_transformer_pred[:plot_n],
    label="Transformer",
    linewidth=1
)

plt.plot(
    pv_test_timestamps.iloc[:plot_n],
    pv_persistence_pred[:plot_n],
    label="Persistence",
    linewidth=1
)

plt.xlabel("Time")
plt.ylabel("PV generation (kWh)")
plt.title("PV Forecasting – Transformer vs Persistence")
plt.legend()

plt.tight_layout()
plt.show()


# 6. Grid-import Transformer forecasting

Grid import is the **target task** for the later transfer-learning experiment.


In [ ]:
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

grid_transformer, grid_transformer_history, grid_transformer_true, grid_transformer_pred = (
    run_transformer_experiment(
        data,
        target="grid_import_hourly",
        lookback=LOOKBACK,
        epochs=30
    )
)

print()

print_metrics(
    "Transformer – Grid-import test set",
    grid_transformer_true,
    grid_transformer_pred
)


In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(
    grid_transformer_history["train_loss"],
    label="Training loss"
)

plt.plot(
    grid_transformer_history["val_loss"],
    label="Validation loss"
)

plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.title("Transformer Training History – Grid Import")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
grid_test_timestamps = test["timestamp"].reset_index(drop=True)
plot_n = min(168, len(grid_transformer_true))

plt.figure(figsize=(15, 5))

plt.plot(
    grid_test_timestamps.iloc[:plot_n],
    grid_transformer_true[:plot_n],
    label="Actual",
    linewidth=1.2
)

plt.plot(
    grid_test_timestamps.iloc[:plot_n],
    grid_transformer_pred[:plot_n],
    label="Transformer",
    linewidth=1
)

plt.plot(
    grid_test_timestamps.iloc[:plot_n],
    grid_persistence_pred[:plot_n],
    label="Persistence",
    linewidth=1
)

plt.xlabel("Time")
plt.ylabel("Grid-import energy (kWh)")
plt.title("Grid-Import Forecasting – Transformer vs Persistence")
plt.legend()

plt.tight_layout()
plt.show()


## 7. Transformer results

The first table compares the Transformer with persistence.

The LSTM values are included separately as a reference from the completed
Notebook 02 run; the LSTM is **not retrained in this notebook**.


In [ ]:
pv_t = regression_metrics(
    pv_transformer_true,
    pv_transformer_pred
)

pv_p = regression_metrics(
    pv_persistence_true,
    pv_persistence_pred
)

grid_t = regression_metrics(
    grid_transformer_true,
    grid_transformer_pred
)

grid_p = regression_metrics(
    grid_persistence_true,
    grid_persistence_pred
)

transformer_results = pd.DataFrame({
    "Task": ["PV generation", "Grid import"],
    "Persistence MAE": [pv_p["MAE"], grid_p["MAE"]],
    "Transformer MAE": [pv_t["MAE"], grid_t["MAE"]],
    "Persistence RMSE": [pv_p["RMSE"], grid_p["RMSE"]],
    "Transformer RMSE": [pv_t["RMSE"], grid_t["RMSE"]],
    "Persistence nRMSE": [pv_p["nRMSE"], grid_p["nRMSE"]],
    "Transformer nRMSE": [pv_t["nRMSE"], grid_t["nRMSE"]]
})

transformer_results


In [ ]:
transformer_results["MAE improvement (%)"] = (
    100
    * (
        transformer_results["Persistence MAE"]
        - transformer_results["Transformer MAE"]
    )
    / transformer_results["Persistence MAE"]
)

transformer_results["RMSE improvement (%)"] = (
    100
    * (
        transformer_results["Persistence RMSE"]
        - transformer_results["Transformer RMSE"]
    )
    / transformer_results["Persistence RMSE"]
)

transformer_results


In [ ]:
# Notebook 02 LSTM benchmark from the completed run.
# These values are reference values, not newly trained here.

lstm_benchmark = pd.DataFrame({
    "Task": ["PV generation", "Grid import"],
    "LSTM MAE": [0.256717, 0.281353],
    "LSTM RMSE": [0.500528, 0.397823]
})

architecture_comparison = pd.DataFrame({
    "Task": ["PV generation", "Grid import"],
    "LSTM MAE": lstm_benchmark["LSTM MAE"],
    "Transformer MAE": [pv_t["MAE"], grid_t["MAE"]],
    "LSTM RMSE": lstm_benchmark["LSTM RMSE"],
    "Transformer RMSE": [pv_t["RMSE"], grid_t["RMSE"]]
})

architecture_comparison


# Conclusions

Notebook 03 adds **Transformer-based time-series forecasting in PyTorch** to
the project.

### Completed

- PyTorch implementation
- Positional encoding
- Transformer Encoder
- 24-hour forecasting window
- Leakage-safe scaling
- Chronological train/validation/test split
- Persistence benchmark
- PV forecasting
- Grid-import forecasting
- MAE, RMSE and nRMSE evaluation
- Comparison with the Notebook 02 LSTM benchmark

### Next step

**Notebook 04: Transfer learning**

The main research experiment will transfer forecasting knowledge from the
PV source task to the grid-import target task and evaluate whether transfer
learning helps when only a limited amount of target-task training data is
available.
